In [1]:
# 示例：如何使用 RRTConfig.rrt_once
# 注意：替换 load_env 的路径与参数为你的环境文件
from rrt_bc import RRTConfig
from crane_env_pyo3 import load_env, Crane
import crane_env_pyo3
import time
import math

C:\Users\yaoyj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
def load_model(path: str, width: int):
    import torch
    from bc_only import NLP, NormalizedEnv, CraneLiftEnv

    ev = NormalizedEnv("envs/测试场景_复杂.obj", [0, 0, 2], 10, 0.2, crane_min_string_length=6)
    # TensorBoard: create writer for environment logging
    env = CraneLiftEnv(ev, max_steps=1000, tb_writer=None, log_tag="Env")

    # 使用默认的MlpPolicy
    model = NLP(env.reset()[0].shape[0], hidden_dim=[width])
    model.load_state_dict(torch.load(path))
    return model

In [ ]:
# 对比加入BC模型与不加入BC模型的参数敏感度

import random


output1 = []
output2 = []
# 加载环境（示例参数）
env = load_env("envs/测试场景.obj", 200, 5.0, 5.0, 0.1, 0.5)

# 创建 Crane（基座位置, yaw_offs, length）
crane = Crane([0.0, 0.0, 2.0], 0.2, 10.0)


# 构造 RRT 配置
cfg = RRTConfig(
    rrt_step_length=crane.crane_length() * 0.03,
    rrt_keep_action_strength=5.0,
    rrt_goal_dist=1.0,
    rrt_max_sample_attempts=500,
    rrt_initial_action=2,
    model=load_model("models/lr050.0__131_w512/bc_model.pth", 512),
    select_model_rate=0.75,
)


start = [3.8472795677099914, -3.8182133490736754, 0.8938367302468908]
end = [6.914603071031225, 4.66693452048438, 6.339378690785946]

start = [-6.188127337130218, 6.729672506700874, 5.809532615266724]
end = [1.6153030860458646, -5.93788534204111, 6.248250039828461]

start = [-0.5555013230583521, -3.4157704037600753, 8.750218269757319]
end = [3.4052199322665793, 2.3659956671349622, 9.116254540559838]


def invalidate_point():
    global start, end
    start = [0.2, 0.2, 0.0]
    end = [1.0, 1.0, 0.0]


invalidate_point()

success_rate_list1 = [] # rrtbc
success_rate_list2 = [] # 改进rrt
success_rate_list3 = [] # uniform rrt

avg_turn_count_list1 = [] # rrtbc
avg_turn_count_list2 = [] # 改进rrt
avg_turn_count_list3 = [] # uniform rrt

avg_time_list1 = [] # rrtbc
avg_time_list2 = [] # 改进rrt
avg_time_list3 = [] # uniform rrt

min_turn_count_list1 = []
min_turn_count_list2 = []
min_turn_count_list3 = []

for path_idx in range(200):

    epochs1 = 50
    epochs2 = 50
    epochs3 = 50
    attempt_1 = []
    print(f"\n=== Path attempt {path_idx} ===")
    invalidate_point()
    while True:
        fail = False
        fail = fail or env.collide(start)
        fail = fail or env.collide(end)
        fail = fail or not crane.valid(env, start)
        fail = fail or not crane.valid(env, end)
        if not fail:
            break
        start = env.random_point()
        end = env.random_point()
    print("start=", start)
    print("end=", end)
    while True:
        cfg.reset_config(
            rrt_step_length=crane.crane_length() * random.uniform(0.005, 0.05), 
            rrt_keep_action_strength=random.uniform(0.01, 1000.0), 
            rrt_max_sample_attempts=random.randint(100, 2000), 
            rrt_initial_action=random.randint(0, 2) 
        )
        path_length = math.sqrt(sum((s - e) ** 2 for s, e in zip(start, end)))
        if not path_length > cfg.rrt_step_length * cfg.rrt_max_sample_attempts:
            break
        
    # 准备好了config，准备开始运行
    ######################################
    # rrtbc
    print("step_length =", cfg.rrt_step_length)
    print("width of env:", env.max()[0] - env.min()[0])
    success1 = 0
    cfg.enable_bc(True)  # 启用 BC 模型
    print(cfg.select_model_rate)
    cfg.uniform_rrt = False

    start_time = time.time()
    # 运行 rrt_once
    for epoch in range(epochs1):
        path = cfg.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
            pass
        else:
            success1 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_1.append((length, turn_count))

    # 显示每次平均时间
    print(f"Average time per epoch: {(time.time() - start_time) / epochs1:.4f} seconds")
    avgtime1 = (time.time() - start_time) / epochs1
    avg_turn_count1 = -1
    for length, turn_count in attempt_1:
        avg_turn_count1 += turn_count
    avg_turn_count1 /= len(attempt_1) if attempt_1 else -1
    min_turn_count1 = -1
    for length, turn_count in attempt_1:
        if min_turn_count1 == -1 or turn_count < min_turn_count1:
            min_turn_count1 = turn_count

    ########################################
    # 改进RRT
    print("\nUsing another RRTConfig instance:\n")
    cfg.enable_bc(False)  # 关闭 BC 模型
    cfg.uniform_rrt = False 

    attempt_2 = []

    start_time = time.time()

    success2 = 0
    # 运行 rrt_once
    for epoch in range(epochs2):
        path = cfg.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
        else:
            success2 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_2.append((length, turn_count))
            
    print(f"Average time per epoch: {(time.time() - start_time) / epochs2:.4f} seconds")
    avgtime2 = (time.time() - start_time) / epochs2
    avg_turn_count2 = -1
    for length, turn_count in attempt_2:
        avg_turn_count2 += turn_count
    avg_turn_count2 /= len(attempt_2) if attempt_2 else -1
    min_turn_count2 = -1
    for length, turn_count in attempt_2:
        if min_turn_count2 == -1 or turn_count < min_turn_count2:
            min_turn_count2 = turn_count

    ##########################################
    # uniform rrt
    print("\nUsing UNIFORM RRTConfig instance:\n")
    cfg.enable_bc(False)  # 关闭 BC 模型
    cfg.uniform_rrt = True 

    attempt_3 = []

    start_time = time.time()

    success3 = 0
    # 运行 rrt_once
    for epoch in range(epochs3):
        path = cfg.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
        else:
            success3 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_3.append((length, turn_count))
            
    print(f"Average time per epoch: {(time.time() - start_time) / epochs2:.4f} seconds")
    avgtime3 = (time.time() - start_time) / epochs2

    avg_turn_count3 = -1
    for length, turn_count in attempt_3:
        avg_turn_count3 += turn_count
    avg_turn_count3 /= len(attempt_3) if attempt_3 else -1
    
    min_turn_count3 = -1
    for length, turn_count in attempt_3:
        if min_turn_count3 == -1 or turn_count < min_turn_count3:
            min_turn_count3 = turn_count

    #########################################
    
    # 显示对比效果
    print(f"\nSuccess rates: success1 = {success1/epochs1}, success2 = {success2/epochs2}")
    print(f"Average times: avgtime1 = {avgtime1:.4f} seconds, avgtime2 = {avgtime2:.4f} seconds")
    print(f"success rate compare:{(success1/epochs1)/(success2/epochs2 + 0.000001)}")
    # print(f"Average times compare:{avgtime1/avgtime2}")
    # 生成输出
    

    success_rate_list1.append(success1 / epochs1)
    success_rate_list2.append(success2 / epochs2)
    success_rate_list3.append(success3 / epochs2)
    avg_turn_count_list1.append(avg_turn_count1)
    avg_turn_count_list2.append(avg_turn_count2)
    avg_turn_count_list3.append(avg_turn_count3)
    avg_time_list1.append(avgtime1)
    avg_time_list2.append(avgtime2)
    avg_time_list3.append(avgtime3)
    min_turn_count_list1.append(min_turn_count1)
    min_turn_count_list2.append(min_turn_count2)
    min_turn_count_list3.append(min_turn_count3)
print(success_rate_list1)
print(success_rate_list2)
print(success_rate_list3)
print()
print(avg_turn_count_list1)
print(avg_turn_count_list2)
print(avg_turn_count_list3)
print()
print(avg_time_list1)
print(avg_time_list2)
print(avg_time_list3)
print()
print(min_turn_count_list1)
print(min_turn_count_list2)
print(min_turn_count_list3)



=== Path attempt 0 ===
start= [-0.3734900844421025, 2.234861833629939, 7.0127119297832134]
end= [-5.4747361195659305, 1.6081819190811046, 6.8515441975196865]
step_length = 0.34215985639135094
width of env: 15.076403617858885
0.75
epoch 0: Found path with 21 waypoints, turn count: 0
epoch 1: Found path with 23 waypoints, turn count: 0
epoch 2: Found path with 27 waypoints, turn count: 0
epoch 3: Found path with 25 waypoints, turn count: 0
epoch 4: Found path with 21 waypoints, turn count: 0
epoch 5: Found path with 26 waypoints, turn count: 0
epoch 6: Found path with 22 waypoints, turn count: 0
epoch 7: Found path with 27 waypoints, turn count: 0
epoch 8: Found path with 23 waypoints, turn count: 0
epoch 9: Found path with 21 waypoints, turn count: 0
epoch 10: Found path with 27 waypoints, turn count: 0
epoch 11: Found path with 25 waypoints, turn count: 0
epoch 12: Found path with 24 waypoints, turn count: 0
epoch 13: Found path with 22 waypoints, turn count: 0
epoch 14: Found path wi

In [7]:
print("success_rate_list1 = ", success_rate_list1)
print("success_rate_list2 = ", success_rate_list2)
print("success_rate_list3 = ", success_rate_list3)
print()
print("avg_turn_count_list1 = ", avg_turn_count_list1)
print("avg_turn_count_list2 = ", avg_turn_count_list2)
print("avg_turn_count_list3 = ", avg_turn_count_list3)
print()
print("avg_time_list1 = ", avg_time_list1)
print("avg_time_list2 = ", avg_time_list2)
print("avg_time_list3 = ", avg_time_list3)
print()
print("min_turn_count_list1 = ", min_turn_count_list1)
print("min_turn_count_list2 = ", min_turn_count_list2)
print("min_turn_count_list3 = ", min_turn_count_list3)


success_rate_list1 =  [1.0, 0.98, 0.7, 0.44, 1.0, 1.0, 1.0, 0.12, 0.0, 1.0, 0.28, 1.0, 0.98, 0.84, 0.0, 1.0, 1.0, 1.0, 1.0, 0.8, 0.0, 1.0, 1.0, 0.94, 1.0, 1.0, 0.56, 0.48, 0.54, 1.0, 1.0, 1.0, 1.0, 0.0, 0.38, 0.52, 0.0, 0.42, 0.6, 0.46, 0.92, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.94, 1.0, 1.0, 1.0, 0.0, 0.08, 0.0, 0.0, 0.1, 0.9, 0.0, 1.0, 0.4, 0.14, 1.0, 1.0, 0.0, 0.92, 1.0, 1.0, 0.78, 0.0, 0.06, 0.9, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.08, 0.98, 0.52, 1.0, 0.78, 1.0, 0.46, 0.0, 0.6, 0.0, 0.0, 0.84, 0.84, 0.0, 1.0, 1.0, 0.08, 0.98, 0.02, 0.58, 1.0, 1.0, 0.4, 0.14, 0.0, 1.0, 1.0, 0.64, 0.0, 0.96, 1.0, 1.0, 0.0, 1.0, 1.0, 0.9, 0.98, 1.0, 1.0, 0.14, 1.0, 1.0, 0.0, 0.68, 0.2, 1.0, 0.72, 0.98, 1.0, 1.0, 1.0, 1.0, 0.9, 0.86, 1.0, 1.0, 0.62, 1.0, 0.68, 0.0, 0.98, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.88, 1.0, 1.0, 1.0, 0.34, 1.0, 0.1, 1.0, 1.0, 1.0, 0.92, 0.0, 0.72, 1.0, 0.88, 0.0, 1.0, 0.88, 0.0, 0.0, 0.92, 1.0, 1.0, 0.98, 0.0, 0.86, 0.62, 0.74, 0.58, 0.96, 1.0, 1.0, 1.0, 1.0, 0.0, 

In [5]:
raise RuntimeError("stop running")

RuntimeError: stop running

In [ ]:
# 对比加入BC模型与不加入BC模型的效果

output1 = []
output2 = []
# 加载环境（示例参数）
env = load_env("envs/测试场景.obj", 200, 5.0, 5.0, 0.1, 0.5)

# 创建 Crane（基座位置, yaw_offs, length）
crane = Crane([0.0, 0.0, 2.0], 0.2, 10.0)


# 构造 RRT 配置
cfg = RRTConfig(
    rrt_step_length=crane.crane_length() * 0.01,
    rrt_keep_action_strength=2.0,
    rrt_goal_dist=1.0,
    rrt_max_sample_attempts=1500,
    rrt_initial_action=0,
    model=load_model("models/lr050.0__131_w512/bc_model.pth", 512),
    select_model_rate=0.3,
)


start = [3.8472795677099914, -3.8182133490736754, 0.8938367302468908]
end = [6.914603071031225, 4.66693452048438, 6.339378690785946]

start = [-6.188127337130218, 6.729672506700874, 5.809532615266724]
end = [1.6153030860458646, -5.93788534204111, 6.248250039828461]

start = [-0.5555013230583521, -3.4157704037600753, 8.750218269757319]
end = [3.4052199322665793, 2.3659956671349622, 9.116254540559838]


def invalidate_point():
    global start, end
    start = [0.2, 0.2, 0.0]
    end = [1.0, 1.0, 0.0]


invalidate_point()

success_rate_list1 = []
success_rate_list2 = []
for path_idx in range(200):
    epochs1 = 50
    epochs2 = 50
    attempt_1 = []
    print(f"\n=== Path attempt {path_idx} ===")
    invalidate_point()
    while True:
        fail = False
        fail = fail or env.collide(start)
        fail = fail or env.collide(end)
        fail = fail or not crane.valid(env, start)
        fail = fail or not crane.valid(env, end)
        if not fail:
            break
        start = env.random_point()
        end = env.random_point()
    print("start=", start)
    print("end=", end)

    print("step_length =", cfg.rrt_step_length)
    print("width of env:", env.max()[0] - env.min()[0])
    success1 = 0
    start_time = time.time()
    # 运行 rrt_once
    for epoch in range(epochs1):
        path = cfg.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
            pass
        else:
            success1 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_1.append((length, turn_count))

    # 显示每次平均时间
    print(f"Average time per epoch: {(time.time() - start_time) / epochs1:.4f} seconds")
    avgtime1 = (time.time() - start_time) / epochs1

    print("\nUsing another RRTConfig instance:\n")
    rrtc2 = crane_env_pyo3.RRTConfig(
        cfg.rrt_step_length,
        cfg.rrt_keep_action_strength,
        cfg.rrt_goal_dist,
        10.0,
        100,
        cfg.rrt_max_sample_attempts,
        cfg.rrt_initial_action,
    )
    attempt_2 = []

    start_time = time.time()

    success2 = 0
    # 运行 rrt_once
    for epoch in range(epochs2):
        path = rrtc2.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
        else:
            success2 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_2.append((length, turn_count))
    # 显示对比效果
    print(f"Average time per epoch: {(time.time() - start_time) / epochs2:.4f} seconds")
    avgtime2 = (time.time() - start_time) / epochs2
    print(f"\nSuccess rates: success1 = {success1/epochs1}, success2 = {success2/epochs2}")
    print(f"Average times: avgtime1 = {avgtime1:.4f} seconds, avgtime2 = {avgtime2:.4f} seconds")
    print(f"success rate compare:{(success1/epochs1)/(success2/epochs2 + 0.000001)}")
    print(f"Average times compare:{avgtime1/avgtime2}")
    success_rate_list1.append(success1 / epochs1)
    success_rate_list2.append(success2 / epochs2)
    output1.append((success1 / epochs1, attempt_1))
    output2.append((success2 / epochs2, attempt_2))
print(success_rate_list1)
print(success_rate_list2)



=== Path attempt 0 ===
start= [1.895481242395844, 3.3508852339339033, 2.5371261930291578]
end= [-5.969022452148094, 2.420327711054556, 1.0470495958221466]
step_length = 0.1
width of env: 15.076403617858885
epoch 0: Found path with 113 waypoints, turn count: 7
epoch 1: Found path with 107 waypoints, turn count: 15


In [ ]:
print(output1)
print(output2)

[(0.0, []), (0.02, [(118, 2)]), (0.98, [(132, 7), (139, 8), (131, 7), (137, 13), (137, 20), (129, 13), (145, 7), (137, 9), (129, 16), (131, 14), (138, 8), (128, 15), (136, 13), (129, 3), (138, 15), (135, 7), (140, 11), (130, 11), (136, 8), (146, 14), (127, 9), (142, 14), (138, 10), (130, 7), (132, 9), (133, 8), (130, 13), (137, 6), (136, 8), (139, 13), (133, 8), (132, 11), (143, 9), (132, 10), (138, 9), (139, 11), (129, 13), (127, 11), (127, 17), (129, 17), (134, 9), (136, 10), (136, 4), (132, 14), (131, 16), (137, 5), (132, 13), (136, 11), (131, 21)]), (0.84, [(77, 2), (83, 6), (67, 2), (74, 2), (70, 5), (79, 8), (74, 0), (77, 2), (69, 0), (73, 0), (82, 4), (75, 0), (79, 2), (75, 0), (76, 2), (76, 4), (80, 3), (77, 2), (75, 2), (73, 0), (70, 0), (74, 5), (70, 0), (73, 2), (67, 0), (74, 0), (81, 6), (75, 4), (80, 4), (74, 2), (72, 2), (87, 2), (74, 0), (76, 3), (81, 8), (87, 6), (71, 2), (77, 2), (73, 0), (76, 0), (69, 0), (74, 0)]), (1.0, [(143, 11), (150, 10), (147, 10), (149, 4), (1

In [ ]:
raise RuntimeError("stop running")

RuntimeError: stop running

In [ ]:
success_rate_list_cmp = [(s1 + 0.01) / (s2 + 0.01) for s1, s2 in zip(success_rate_list1, success_rate_list2)]
print("Success rate comparisons:", success_rate_list_cmp)

Success rate comparisons: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0000000000000004, 9.0, 1.041237113402062, 1.0, 1.0, 0.5918367346938777, 1.0, 3.2222222222222228, 1.0, 1.041237113402062, 19.0, 1.02020202020202, 1.0, 1.0, 1.0, 0.5999999999999999, 1.0, 1.4057971014492752, 1.0, 1.1632653061224492, 6.199999999999999, 1.0, 1.021505376344086, 1.0, 1.0, 1.063157894736842, 1.0879120879120878, 1.0, 1.0, 2.5999999999999996, 1.0459770114942528, 1.2784810126582278, 1.2666666666666664, 1.0, 21.0, 1.9743589743589742, 1.041237113402062, 1.0, 1.0, 1.0, 1.0, 15.000000000000002, 1.216867469879518, 1.0, 1.0, 1.0, 1.0, 1.0, 5.0, 1.0, 1.0, 1.0, 0.33333333333333337, 1.0, 1.0, 1.2933333333333332, 1.0, 1.3396226415094339, 1.0, 0.9012345679012345, 1.0, 1.160919540229885, 1.0, 1.0, 3.0, 6.6, 0.974025974025974, 1.0, 1.063157894736842, 1.6222222222222222, 1.0, 1.0, 1.8363636363636362, 0.4285714285714286, 0.9801980198019802, 1.0, 15.000000000000002, 1.0, 1.0, 1.0421052631578949, 1.041237113402062, 0.9797979797979798, 1.2

In [ ]:
for i in range(len(success_rate_list_cmp)):
    if success_rate_list1[i] == success_rate_list2[i] and (success_rate_list1[i] == 0.0 or success_rate_list1[i] == 1.0):
        continue
    print(success_rate_list_cmp[i],end=' ')

3.0000000000000004 9.0 1.041237113402062 0.5918367346938777 3.2222222222222228 1.041237113402062 19.0 1.02020202020202 0.5999999999999999 1.4057971014492752 1.1632653061224492 6.199999999999999 1.021505376344086 1.063157894736842 1.0879120879120878 2.5999999999999996 1.0459770114942528 1.2784810126582278 1.2666666666666664 21.0 1.9743589743589742 1.041237113402062 1.0 15.000000000000002 1.216867469879518 1.0 5.0 0.33333333333333337 1.2933333333333332 1.3396226415094339 0.9012345679012345 1.160919540229885 3.0 6.6 0.974025974025974 1.063157894736842 1.6222222222222222 1.8363636363636362 0.4285714285714286 0.9801980198019802 15.000000000000002 1.0421052631578949 1.041237113402062 0.9797979797979798 1.2191780821917808 19.0 1.4615384615384617 

In [ ]:
print(success_rate_list1)
print(success_rate_list2)

[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.14, 0.08, 1.0, 1.0, 1.0, 0.28, 0.0, 0.28, 1.0, 1.0, 0.18, 1.0, 0.0, 1.0, 0.0, 0.08, 0.0, 0.96, 1.0, 0.56, 0.3, 0.0, 0.94, 1.0, 1.0, 1.0, 0.98, 1.0, 0.0, 0.38, 0.9, 1.0, 0.18, 1.0, 0.2, 0.76, 1.0, 0.0, 0.08, 0.0, 0.0, 0.14, 1.0, 0.98, 1.0, 0.0, 1.0, 0.0, 0.04, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.96, 0.0, 0.7, 1.0, 0.72, 1.0, 1.0, 0.0, 0.0, 0.02, 0.32, 0.74, 1.0, 1.0, 0.72, 0.0, 0.0, 1.0, 0.02, 0.98, 0.0, 0.44, 1.0, 1.0, 0.98, 1.0, 0.96, 0.88, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.18, 1.0, 0.56, 1.0]
[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.04, 0.0, 0.96, 1.0, 1.0, 0.48, 0.0, 0.08, 1.0, 0.96, 0.0, 0.98, 0.0, 1.0, 0.0, 0.14, 0.0, 0.68, 1.0, 0.48, 0.04, 0.0, 0.92, 1.0, 1.0, 0.94, 0.9, 1.0, 0.0, 0.14, 0.86, 0.78, 0.14, 1.0, 0.0, 0.38, 0.96, 0.0, 0.08, 0.0, 0.0, 0.0, 0.82, 0.98, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.02, 0.0, 1.0, 0.74, 0.0, 0.52, 1.0, 0.8, 1.0, 0.86, 0.0, 0.0, 0.0, 0.04, 0.76, 1.0, 0.94, 0.44, 0.0, 0.0, 0.54, 0.06, 1.0, 0.0, 0.02, 1.0, 1.0, 0.94, 0.96

In [ ]:
epc1 = 50
epc2 = 50
cmp_list = [
    ((s1*epc1 + 1)/(epc1+2)) / ((s2*epc2 + 1)/(epc2+2)) 
    for s1, s2 in zip(success_rate_list1, success_rate_list2)
]
print(cmp_list)

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.6666666666666665, 5.0, 1.0408163265306123, 1.0, 1.0, 0.6000000000000001, 1.0, 3.0, 1.0, 1.0408163265306123, 10.0, 1.02, 1.0, 1.0, 1.0, 0.625, 1.0, 1.4, 1.0, 1.16, 5.333333333333333, 1.0, 1.021276595744681, 1.0, 1.0, 1.0625, 1.0869565217391306, 1.0, 1.0, 2.5, 1.0454545454545454, 1.275, 1.25, 1.0, 11.0, 1.95, 1.0408163265306123, 1.0, 1.0, 1.0, 1.0, 8.0, 1.2142857142857142, 1.0, 1.0, 1.0, 1.0, 1.0, 3.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.2894736842105263, 1.0, 1.3333333333333333, 1.0, 0.9024390243902439, 1.0, 1.159090909090909, 1.0, 1.0, 2.0, 5.666666666666666, 0.9743589743589743, 1.0, 1.0625, 1.608695652173913, 1.0, 1.0, 1.8214285714285714, 0.5, 0.9803921568627452, 1.0, 11.499999999999998, 1.0, 1.0, 1.0416666666666667, 1.0408163265306123, 0.98, 1.2162162162162162, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 10.0, 1.0, 1.45, 1.0]


In [ ]:
a = zip(success_rate_list1, success_rate_list2)
a = filter(lambda x: not (x[0] == x[1] and (x[0] == 0 or x[1] == 1)), a)
# cmp_list = [1 if s1 > s2 else 0.5 if s1 == s2 else 0 for s1, s2 in a]
cmp_list = [((s1 * epc1 + 1) / (epc1 + 2)) / ((s2 * epc2 + 1) / (epc2 + 2)) for s1, s2 in a]
print(cmp_list)

[2.6666666666666665, 5.0, 1.0408163265306123, 0.6000000000000001, 3.0, 1.0408163265306123, 10.0, 1.02, 0.625, 1.4, 1.16, 5.333333333333333, 1.021276595744681, 1.0625, 1.0869565217391306, 2.5, 1.0454545454545454, 1.275, 1.25, 11.0, 1.95, 1.0408163265306123, 1.0, 8.0, 1.2142857142857142, 1.0, 3.0, 0.5, 1.2894736842105263, 1.3333333333333333, 0.9024390243902439, 1.159090909090909, 2.0, 5.666666666666666, 0.9743589743589743, 1.0625, 1.608695652173913, 1.8214285714285714, 0.5, 0.9803921568627452, 11.499999999999998, 1.0416666666666667, 1.0408163265306123, 0.98, 1.2162162162162162, 10.0, 1.45]


In [ ]:
# 对比加入BC和随机采样的效果
output1 = []
output2 = []
# 加载环境（示例参数）
env = load_env("envs/测试场景.obj", 200, 5.0, 5.0, 0.1, 0.5)

# 创建 Crane（基座位置, yaw_offs, length）
crane = Crane([0.0, 0.0, 2.0], 0.2, 10.0)


# 构造 RRT 配置
cfg = RRTConfig(
    rrt_step_length=crane.crane_length() * 0.01,
    rrt_keep_action_strength=2.0,
    rrt_goal_dist=1.0,
    rrt_max_sample_attempts=1500,
    rrt_initial_action=0,
    model=load_model("models/lr050.0__131_w512/bc_model.pth", 512),
    select_model_rate=0.3,
)
rrtc2 = RRTConfig(
    rrt_step_length=crane.crane_length() * 0.01,
    rrt_keep_action_strength=2.0,
    rrt_goal_dist=1.0,
    rrt_max_sample_attempts=1500,
    rrt_initial_action=0,
    model=load_model("models/lr050.0__131_w512/bc_model.pth", 512),
    select_model_rate=0.3,
    apply_random=True,
)

start = [3.8472795677099914, -3.8182133490736754, 0.8938367302468908]
end = [6.914603071031225, 4.66693452048438, 6.339378690785946]

start = [-6.188127337130218, 6.729672506700874, 5.809532615266724]
end = [1.6153030860458646, -5.93788534204111, 6.248250039828461]

start = [-0.5555013230583521, -3.4157704037600753, 8.750218269757319]
end = [3.4052199322665793, 2.3659956671349622, 9.116254540559838]


def invalidate_point():
    global start, end
    start = [0.2, 0.2, 0.0]
    end = [1.0, 1.0, 0.0]


invalidate_point()

success_rate_list1 = []
success_rate_list2 = []
for path_idx in range(100):
    epochs1 = 50
    epochs2 = 50
    attempt_1 = []
    print(f"\n=== Path attempt {path_idx} ===")
    invalidate_point()
    while True:
        fail = False
        fail = fail or env.collide(start)
        fail = fail or env.collide(end)
        fail = fail or not crane.valid(env, start)
        fail = fail or not crane.valid(env, end)
        if not fail:
            break
        start = env.random_point()
        end = env.random_point()
    print("start=", start)
    print("end=", end)

    print("step_length =", cfg.rrt_step_length)
    print("width of env:", env.max()[0] - env.min()[0])
    success1 = 0
    start_time = time.time()
    # 运行 rrt_once
    for epoch in range(epochs1):
        path = cfg.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
            pass
        else:
            success1 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_1.append((length, turn_count))

    # 显示每次平均时间
    print(f"Average time per epoch: {(time.time() - start_time) / epochs1:.4f} seconds")
    avgtime1 = (time.time() - start_time) / epochs1

    print("\nUsing another RRTConfig instance:\n")
   
    attempt_2 = []

    start_time = time.time()

    success2 = 0
    # 运行 rrt_once
    for epoch in range(epochs2):
        path = rrtc2.rrt_once(crane, env, start, end)
        if path is None:
            print(f"epoch {epoch}: No path found")
        else:
            success2 += 1
            length = len(path)
            _last_idx = path[-1][1]
            turn_count = 0
            for p, move_idx, step in path:
                # print(p, move_idx, step)
                if move_idx != _last_idx:
                    turn_count += 1
                _last_idx = move_idx
            print(f"epoch {epoch}: Found path with {length} waypoints, turn count: {turn_count}")
            attempt_2.append((length, turn_count))
    # 显示对比效果
    print(f"Average time per epoch: {(time.time() - start_time) / epochs2:.4f} seconds")
    avgtime2 = (time.time() - start_time) / epochs2
    print(f"\nSuccess rates: success1 = {success1/epochs1}, success2 = {success2/epochs2}")
    print(f"Average times: avgtime1 = {avgtime1:.4f} seconds, avgtime2 = {avgtime2:.4f} seconds")
    print(f"success rate compare:{(success1/epochs1)/(success2/epochs2 + 0.000001)}")
    print(f"Average times compare:{avgtime1/avgtime2}")
    success_rate_list1.append(success1 / epochs1)
    success_rate_list2.append(success2 / epochs2)
    output1.append((success1 / epochs1, attempt_1))
    output2.append((success2 / epochs2, attempt_2))
print(success_rate_list1)
print(success_rate_list2)



=== Path attempt 0 ===
start= [6.427034042120006, 2.0332866827100755, 1.347125146321745]
end= [-5.891072620813708, 1.3735362580948731, 9.9421981058643]
step_length = 0.1
width of env: 15.076403617858885
epoch 0: No path found
epoch 1: No path found
epoch 2: No path found
epoch 3: No path found
epoch 4: No path found
epoch 5: No path found
epoch 6: No path found
epoch 7: No path found
epoch 8: No path found
epoch 9: No path found
epoch 10: No path found
epoch 11: No path found
epoch 12: No path found
epoch 13: No path found
epoch 14: No path found
epoch 15: No path found
epoch 16: No path found
epoch 17: No path found
epoch 18: No path found
epoch 19: No path found
epoch 20: No path found
epoch 21: No path found
epoch 22: No path found
epoch 23: No path found
epoch 24: No path found
epoch 25: No path found
epoch 26: No path found
epoch 27: No path found
epoch 28: No path found
epoch 29: No path found
epoch 30: No path found
epoch 31: No path found
epoch 32: No path found
epoch 33: No p